# Alpaca Pipeline Check

This notebook fetches:
- Last **day** of **5-minute** bars (indicator tuning)
- Last **week** of **1-hour** bars (support/resistance)


## Required Environment Variables

Set these before running the notebook (prefer `.env` over hardcoding secrets):

- `ALPACA_PAPER_API_KEY`
- `ALPACA_PAPER_SECRET_KEY`
- `PAPER=true`
- `TRADE_API_URL=https://paper-api.alpaca.markets`


In [ ]:
# Optional: set these here for this notebook session.
# Prefer using `fresh_simple_trading_project/.env` (or repo-root `.env`) instead.
ALPACA_PAPER_API_KEY = ""
ALPACA_PAPER_SECRET_KEY = ""
PAPER = "true"
TRADE_API_URL = "https://paper-api.alpaca.markets"


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "fresh_simple_trading_project").exists():
            return candidate
        if (candidate / "fresh_simple_trading_project" / "src" / "fresh_simple_trading_project").exists():
            return candidate / "fresh_simple_trading_project"
    raise RuntimeError("Could not find the fresh_simple_trading_project root from the current working directory.")


project_root = find_project_root()
sys.path.insert(0, str(project_root / "src"))
load_dotenv(project_root.parent / ".env")
load_dotenv(project_root / ".env")

ALPACA_PAPER_API_KEY = (globals().get("ALPACA_PAPER_API_KEY") or os.getenv("ALPACA_PAPER_API_KEY", "")).strip()
ALPACA_PAPER_SECRET_KEY = (globals().get("ALPACA_PAPER_SECRET_KEY") or os.getenv("ALPACA_PAPER_SECRET_KEY", "")).strip()
TRADE_API_URL = (globals().get("TRADE_API_URL") or os.getenv("TRADE_API_URL") or "https://paper-api.alpaca.markets").strip()
PAPER = (globals().get("PAPER") or os.getenv("PAPER") or "true").strip()

os.environ["ALPACA_PAPER_API_KEY"] = ALPACA_PAPER_API_KEY
os.environ["ALPACA_PAPER_SECRET_KEY"] = ALPACA_PAPER_SECRET_KEY
os.environ["TRADE_API_URL"] = TRADE_API_URL
os.environ["PAPER"] = PAPER

from fresh_simple_trading_project.alpaca_integration import AlpacaService
from fresh_simple_trading_project.config import Settings

settings = Settings.from_env(project_root=project_root)
if not settings.alpaca.enabled:
    raise RuntimeError(
        "Missing Alpaca credentials. Set ALPACA_PAPER_API_KEY and ALPACA_PAPER_SECRET_KEY before running this notebook."
    )

service = AlpacaService(settings.alpaca)
symbol = settings.trading.symbol

# Requested windows:
support_resistance_lookback_days = 7  # 1h bars
indicator_lookback_days = 1  # 5m bars

print({
    "project_root": str(project_root),
    "symbol": symbol,
    "support_resistance_lookback_days": support_resistance_lookback_days,
    "indicator_lookback_days": indicator_lookback_days,
    "trade_api_url": TRADE_API_URL,
    "paper_trading": settings.alpaca.paper_trading,
})


In [ ]:
from IPython.display import display

hourly_bars = service.fetch_hourly_bars(symbol, lookback_days=support_resistance_lookback_days)
five_minute_bars = service.fetch_five_minute_bars(symbol, lookback_days=indicator_lookback_days)
latest_trade = service.fetch_latest_trade(symbol)
news_items = service.fetch_recent_news(symbol, days=7, limit=5)
clock = service.get_clock()
buying_power = service.get_buying_power()
position_open, position_qty, avg_entry_price = service.get_open_position(symbol)
account_state = service.get_account_state(symbol)

print(f"Fetched {len(hourly_bars)} 1-hour bars for {symbol}")
print(f"1H window: {hourly_bars.index.min()} -> {hourly_bars.index.max()}")
print(f"Fetched {len(five_minute_bars)} 5-minute bars for {symbol}")
print(f"5M window: {five_minute_bars.index.min()} -> {five_minute_bars.index.max()}")
print(f"Latest trade price: {latest_trade}")
print(f"Clock open: {clock.is_open}")
print(f"Buying power: {buying_power}")
print(f"Position open: {position_open}, qty={position_qty}, avg_entry_price={avg_entry_price}")
print(f"Account state used by the pipeline: {account_state}")
print(f"Recent Alpaca news articles: {len(news_items)}")


In [ ]:
import pandas as pd
from IPython.display import display

print("5-minute candlestick tail (indicator tuning window)")
display(five_minute_bars.tail(20))

print("1-hour candlestick tail (support/resistance window)")
display(hourly_bars.tail(20))

print("Recent Alpaca news")
news_frame = pd.DataFrame(
    [
        {
            "headline": article.headline,
            "summary": article.summary,
            "source": article.source,
            "url": article.url,
            "published_at": article.published_at,
        }
        for article in news_items
    ]
)
display(news_frame)


In [ ]:
from IPython.display import display

from fresh_simple_trading_project.features import FeatureEngineeringModule
from fresh_simple_trading_project.market_analysis import MarketAnalysisModule

feature_engineer = FeatureEngineeringModule(settings.trading)
feature_frame = feature_engineer.build(five_minute_bars)

analysis_module = MarketAnalysisModule(settings.trading)
analysis = analysis_module.analyze(symbol, feature_frame, hourly_bars=hourly_bars)

print("Major support levels (1H):", analysis.support_levels)
print("Major resistance levels (1H):", analysis.resistance_levels)
print("Nearest major support:", analysis.nearest_support)
print("Nearest major resistance:", analysis.nearest_resistance)

print("Indicator snapshot (5M features tail)")
display(
    feature_frame[
        [
            "close",
            "volume",
            "return",
            "ma_short",
            "ma_long",
            "rsi",
            "macd",
            "macd_signal",
            "rolling_volatility",
            "buy_trigger",
            "sell_trigger",
        ]
    ].tail(40)
)
